# Mesh convergence -- $N_x=0$, five meshes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BecerraMiguel/SemiFree-Solver/blob/main/notebooks/02_mesh_convergence_Nx0.ipynb)

The semi-free solver is run on five DIII-D resolutions (82x142, 100x172, 151x261, 213x368, 301x521) and the
convergence of the solution under mesh refinement is studied.

**Analysis:**
1. *Convergence of $\psi$.* The 301x521 mesh is the reference; each of the other four solutions is interpolated
onto it with an order-2 bivariate spline. The error is the $L_2$ (RMS) norm of the difference, and
$\ln e_k=p\ln h_k+c$ is fitted in three regions: the full domain, the plasma interior, and the full domain
excluding a disc of 0.02 m around every coil. The expected order is $p\approx2$ (the order of the Cut-Cell
quadrature), except over the full domain, where the point-like coils degrade it.
2. *Convergence of the coil currents:* relative change between consecutive meshes and an order fit.
3. *Run times.*

**Estimated time (Colab, 2 cores):** ~1.5 min build + per-mesh times (through writing `psi_check.txt`) of
0.7, 1.0, 2.9, 8.6 and 28.6 min = **~45 minutes** in total (the $B_R$ and $B_Z$ outputs are not computed here:
this analysis does not use them and they would triple the run time). Quick mode: ~10 minutes.


**DIII-D case** (same for every mesh): $R_0=1.67$ m, $a=0.67$ m, $\kappa=1.77$, $\delta=0.30$,
$I_p=1.5$ MA, $P_{axis}=50$ kPa, $B_{axis}=2.0$ T, $\Psi_b=0$, with $N_c=18$ PF coils.
Computational domain: $R\in[0.15,\,3.0]$ m, $Z\in[-1.75,\,1.75]$ m.

The inputs ($J_\phi$, boundary, coil positions) are shipped with the repository in `cases/DIII-D_<mesh>/`, and the
solver configuration in `configs/DIII-D_<mesh>.json`. This notebook clones the repository, builds the solver and
runs everything from there: nothing has to be uploaded.


**Options (first code cell):**
- `USE_GOOGLE_DRIVE = True` keeps the results in your Drive: if Colab disconnects, re-running the notebook reuses
the runs that already finished, and the notebooks share results with each other.
- `QUICK_MODE = True` uses only the 82x142, 100x172 and 151x261 meshes.

The values labelled *reference* are those obtained in earlier runs of the same study on Colab (2 cores). Timings
depend on the hardware; $\psi$ and the coil currents should agree with the reference up to rounding.



In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BecerraMiguel/SemiFree-Solver.git'
REPO_REF = None        # branch or tag to clone (None = default branch)
REPO_DIR = os.environ.get('REPRODUCE_REPO_DIR', '/content/SemiFree-Solver')
if not os.path.isdir(REPO_DIR):
    cmd = ['git', 'clone', '--depth', '1'] + (['--branch', REPO_REF] if REPO_REF else [])
    subprocess.run(cmd + [REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/notebooks')
import reproduce_common as rc
import numpy as np
import matplotlib.pyplot as plt

USE_GOOGLE_DRIVE = False   # True: keep results in Google Drive (survive disconnections, shared between notebooks)
QUICK_MODE = False       # True: only the three coarsest meshes (fast test; the order fits then use 2 points)
WORK = rc.default_work_dir(USE_GOOGLE_DRIVE)
os.makedirs(f'{WORK}/figures', exist_ok=True)
TAGS = rc.QUICK_TAGS if QUICK_MODE else rc.MESH_TAGS
print('Meshes:', TAGS)
rc.environment_report()
print('Working directory:', WORK)

## 1. Build
A copy of the solver without the dense $B_R$/$B_Z$ sweeps is compiled (the repository is not modified). $\psi$ and the coil currents are identical to those of the full version.

In [ ]:
BIN = rc.build_semifree(REPO_DIR, WORK, skip_bfield=True)
rc.mesh_table(REPO_DIR, TAGS)

## 2. Run the solver on each mesh
Meshes that already have results in the working directory are reused.

In [ ]:
for tag in TAGS:
    rc.run_semifree(BIN, REPO_DIR, WORK, 'Nx0', tag)

## 3. Load results and run times

In [ ]:
RUNS = rc.load_runs(REPO_DIR, WORK, 'Nx0', TAGS)
rc.print_timing_table(RUNS, TAGS, 'Nx0')

## 4. Convergence of $\psi$
Fitted orders next to the reference values.

In [ ]:
PSI = rc.psi_convergence(REPO_DIR, RUNS, TAGS)
rc.print_psi_table(PSI, 'Nx0')
rc.plot_psi_convergence(PSI, 'Nx=0', fname=f'{WORK}/figures/nx0_psi_convergence.png')
plt.show()

## 5. Convergence of the coil currents

In [ ]:
COIL = rc.coil_convergence(REPO_DIR, RUNS, TAGS)
rc.print_coil_table(COIL, 'Nx0')

## 6. Run times

In [ ]:
series = {'Nx=0 (this run)': {t: r['timing']['psi_min'] for t, r in RUNS.items()},
          'Nx=0 (reference, Colab)': rc.REFERENCE['psi_minutes']['Nx0']}
rc.plot_timing(REPO_DIR, series, fname=f'{WORK}/figures/nx0_timing.png')
plt.show()

## 7. Results bundle (optional)

In [ ]:
import glob
zp = rc.zip_results(WORK, ['Nx0'], f'{WORK}/results_mesh_convergence_Nx0.zip',
                    extra_files=sorted(glob.glob(f'{WORK}/figures/nx0_*.png')))
rc.offer_download(zp)